In [1]:
# Step 1: Installing and importing required libraries

import pandas as pd
from datetime import datetime, timedelta
import os
import subprocess
from feast import FeatureStore, Entity, FeatureView, FileSource, Field, KafkaSource, StreamFeatureView
from feast import ValueType
from feast.data_format import JsonFormat

In [2]:
# Step 2: Creating folder structure for Feature Store

# Create feature repo folder 
if not os.path.exists("feature_repo"):
    os.makedirs("feature_repo")
    
if not os.path.exists("data"):
    os.makedirs("data")

In [3]:
# Step 3: Defining the Entity

# Customer entity (sender)
customer_entity = Entity(
    name="customer_id",
    value_type=ValueType.STRING,
    description="Customer who initiated the transaction",
    join_keys=["nameOrig"]
)

print("✅ Entity defined:")
print(f"   - {customer_entity.name}")

✅ Entity defined:
   - customer_id


In [4]:
# 5. Reading the dataset (only necessary columns for speed)
cols = ['step', 'type', 'amount', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
df_all = pd.read_csv('./dataset/paysim.csv', usecols=cols)

# fraud_df = df_all[df_all['isFraud'] == 1]
# non_fraud_df = df_all[df_all['isFraud'] == 0]


# sample_size = 15000  
df = df_all.sample(n=len(df_all), random_state=42)


print(f"Dataset size: {df.shape}")
print(f"Number of fraud cases: {df['isFraud'].sum()}")
print(f"Number of normal cases: {len(df) - df['isFraud'].sum()}")
print(f"Fraud ratio: {df['isFraud'].mean()*100:.4f}%")
print(f"Unique customers (nameOrig): {df['nameOrig'].nunique()}")

Dataset size: (6362620, 7)
Number of fraud cases: 8213
Number of normal cases: 6354407
Fraud ratio: 0.1291%
Unique customers (nameOrig): 6353307


In [5]:
# 6. Temporal feature engineering
df['timestamp'] = pd.to_datetime(df['step'], unit='h', origin='2024-01-01')
df = df.sort_values(['nameOrig', 'step'])
group = df.groupby('nameOrig')
df['transaction_count_24h'] = group['amount'].cumcount() + 1
df['avg_amount_24h'] = group['amount'].cumsum() / df['transaction_count_24h']
df['amount_zscore'] = (df['amount'] - df['avg_amount_24h']) / (df['amount'].std() + 1e-9)
df['rapid_succession'] = group['step'].diff().le(1).astype(int)
df = df.fillna(0)

feast_df = df[['timestamp', 'nameOrig', 'nameDest', 'step', 'type', 'amount','transaction_count_24h', 'avg_amount_24h', 'amount_zscore','rapid_succession', 'isFraud']].copy()
feast_df['customer_id'] = feast_df['nameOrig'].astype(str)
feast_df['event_timestamp'] = feast_df['timestamp']

if not os.path.exists('data'): os.makedirs('data')
feast_df.to_parquet('data/transactions.parquet', index=False)

print(f"✅ Ready dataset size: {feast_df.shape}")
print(f"   Columns: {list(feast_df.columns)}")

✅ Ready dataset size: (6362620, 13)
   Columns: ['timestamp', 'nameOrig', 'nameDest', 'step', 'type', 'amount', 'transaction_count_24h', 'avg_amount_24h', 'amount_zscore', 'rapid_succession', 'isFraud', 'customer_id', 'event_timestamp']


In [6]:
# Step 7: Defining FileSource and Feature View
from feast.types import Float64, Int32, String ,Float32,Int64
batch_source = FileSource(
    name="transactions_batch_source",
    path="data/transactions.parquet",
    timestamp_field="timestamp",
)

transaction_feature_view = FeatureView(
    name="transaction_features",
    entities=[customer_entity],
    ttl=timedelta(days=30),
    source=batch_source,
    schema=[
        Field(name="step", dtype=Int32),
        Field(name="type", dtype=String),
        Field(name="amount", dtype=Float32),
        Field(name="transaction_count_24h", dtype=Int64),
        Field(name="avg_amount_24h", dtype=Float32),
        Field(name="amount_zscore", dtype=Float32),
        Field(name="rapid_succession", dtype=Int64),
    ],
    online=True,
)

print("✅ FileSource and Feature View defined")
print(f"   Feature View name: {transaction_feature_view.name}")
print(f"   Entities: customer_id, recipient_id")

✅ FileSource and Feature View defined
   Feature View name: transaction_features
   Entities: customer_id, recipient_id


In [7]:
# 8
repo_path = "feature_repo"

definitions_content = """from datetime import timedelta
from feast import Entity, FeatureView, FileSource, Field, ValueType
from feast.types import Float32, Int32, Int64, String

# Entities
customer_entity = Entity(
    name="customer_id",
    join_keys=["customer_id"],
    value_type=ValueType.STRING
)

# Source
source = FileSource(
    path="../data/transactions.parquet",
    timestamp_field="event_timestamp",
)

# Feature View
transaction_features = FeatureView(
    name="transaction_features",
    entities=[customer_entity],
    ttl=timedelta(days=30),
    source=source,
    schema=[
        Field(name="step", dtype=Int32),
        Field(name="type", dtype=String),
        Field(name="amount", dtype=Float32),             
        Field(name="transaction_count_24h", dtype=Int64),
        Field(name="avg_amount_24h", dtype=Float32),
        Field(name="amount_zscore", dtype=Float32),
        Field(name="rapid_succession", dtype=Int64),
    ],
    online=True,
)
"""

with open(os.path.join(repo_path, "definitions.py"), "w", encoding="utf-8") as f:
    f.write(definitions_content)


result = subprocess.run(
    ["feast", "apply"], 
    cwd=repo_path, 
    shell=True, 
    capture_output=True, 
    text=True
)

print(result.stdout)
if result.stderr:
    print("Info/Error:", result.stderr)

No project found in the repository. Using project name paysim_feature_store defined in feature_store.yaml
Applying changes for project paysim_feature_store
Deploying infrastructure for transaction_features



In [8]:
# 9
test_customer = feast_df[feast_df['nameOrig'] == feast_df['nameOrig'].iloc[0]]
file_size = os.path.getsize('data/transactions.parquet') / (1024*1024)

print(test_customer[['step', 'transaction_count_24h']].head(30))

print(f"File size: {file_size:.2f} MB")
print(f"Number of rows: {len(feast_df)}")
print(f"Number of fraud cases: {feast_df['isFraud'].sum()}")
print(f"Number of normal cases: {len(feast_df) - feast_df['isFraud'].sum()}")

         step  transaction_count_24h
3196942   249                      1
File size: 235.12 MB
Number of rows: 6362620
Number of fraud cases: 8213
Number of normal cases: 6354407


In [9]:
# 10)
 
store = FeatureStore(repo_path="feature_repo")

start_date = feast_df['event_timestamp'].min()
end_date = feast_df['event_timestamp'].max()

store.materialize(
                     feature_views=["transaction_features"], 
                     start_date=start_date,
                     end_date=end_date
                    )

print("✅ Materialization tamamlandı")

Materializing 1 feature views from 2024-01-01 01:00:00+00:00 to 2024-01-31 23:00:00+00:00 into the redis online store.

transaction_features:
✅ Materialization tamamlandı


In [10]:
# Step 11: Online store - corrected

store = FeatureStore(repo_path="feature_repo")

# Get customers from DataFrame
real_customers = feast_df['customer_id'].dropna().unique()[:5].tolist()
entity_rows = [{"customer_id": str(cust_id)} for cust_id in real_customers]

try:
    features = store.get_online_features(
        features=[
            "transaction_features:amount",
            "transaction_features:transaction_count_24h",
            "transaction_features:avg_amount_24h",
        ],
        entity_rows=entity_rows,
    ).to_dict()

    print("✅ Feature vectors from online store:")
    print(pd.DataFrame(features))
except Exception as e:
    print(f"❌ Error occurred: {e}")

✅ Feature vectors from online store:
   customer_id  transaction_count_24h  avg_amount_24h         amount
0  C1000000639                      1   244486.453125  244486.453125
1  C1000001337                      1     3170.280029    3170.280029
2  C1000001725                      1     8424.740234    8424.740234
3  C1000002591                      1   261877.187500  261877.187500
4  C1000003372                      1    20528.650391   20528.650391


In [11]:
# Step 12: Offline store - corrected

store = FeatureStore(repo_path="feature_repo")

real_customers = feast_df['customer_id'].dropna().unique()[:10].tolist()

entity_df = pd.DataFrame({
    "customer_id": real_customers,
    "event_timestamp": [feast_df['event_timestamp'].max() - timedelta(days=i) for i in range(len(real_customers))]
})

training_df = store.get_historical_features(
    features=[
        "transaction_features:amount",
        "transaction_features:transaction_count_24h",
        "transaction_features:avg_amount_24h",
    ],
    entity_df=entity_df,
).to_df()

print("✅ Training dataset from offline store:")
print(f"\nDataset size: {training_df.shape}")
training_df

✅ Training dataset from offline store:

Dataset size: (10, 5)


,customer_id,event_timestamp,amount,transaction_count_24h,avg_amount_24h
0,C1000004530,2024-01-24 23:00:00+00:00,93865.13,1,93865.13
1,C1000001725,2024-01-29 23:00:00+00:00,8424.74,1,8424.74
2,C1000003372,2024-01-27 23:00:00+00:00,20528.65,1,20528.65
3,C1000005555,2024-01-22 23:00:00+00:00,233109.79,1,233109.79
4,C1000001337,2024-01-30 23:00:00+00:00,3170.28,1,3170.28
5,C1000005353,2024-01-23 23:00:00+00:00,3228390.11,1,3228390.11
6,C1000002591,2024-01-28 23:00:00+00:00,261877.19,1,261877.19
7,C1000000639,2024-01-31 23:00:00+00:00,244486.46,1,244486.46
8,C1000003615,2024-01-26 23:00:00+00:00,49360.77,1,49360.77
9,C1000004053,2024-01-25 23:00:00+00:00,211189.64,1,211189.64


In [12]:
# Step 13: Streaming - corrected

# Correct entity
customer_entity = Entity(
    name="customer_id", 
    join_keys=["customer_id"],
    value_type=ValueType.STRING
)

# Json format
json_format = JsonFormat(schema_json='{"type": "record", "name": "Transaction", "fields": [{"name": "amount", "type": "float"}, {"name": "transaction_count_24h", "type": "int"}]}')

# Kafka source
kafka_source = KafkaSource(
    name="streaming_transactions",
    timestamp_field="event_timestamp",
    message_format=json_format,
    kafka_bootstrap_servers="localhost:9092",
    topic="transactions_topic",
    batch_source=FileSource(
        path="data/transactions.parquet",
        event_timestamp_column="event_timestamp",
    ),
)

# Stream feature view
stream_feature_view = StreamFeatureView(
    name="streaming_transaction_features",
    entities=[customer_entity],
    ttl=timedelta(hours=24),
    source=kafka_source,
    schema=[
        Field(name="amount", dtype=Float32),
        Field(name="transaction_count_24h", dtype=Int64),
    ],
    online=True,
)

c:\Users\USER\Documents\Devlab_intership\Projects\Query a Feature Store for ML Training & Serving\feast_env\lib\site-packages\feast\stream_feature_view.py:125: RuntimeWarning: Stream feature views are experimental features in alpha development. Some functionality may still be unstable so functionality can change in the future.
  warnings.warn(


In [13]:
# Step 14: Feature monitoring alerts - FIXED

class FeatureMonitor:
    
    def __init__(self, store, feature_view_name, feature_name, threshold=0.3):
        self.store = store
        self.feature_view_name = feature_view_name
        self.feature_name = feature_name
        self.threshold = threshold
        self.alerts = []
    
    def check_feature_stats(self, entity_rows):
        try:
            features = self.store.get_online_features(
                features=[f"{self.feature_view_name}:{self.feature_name}"],
                entity_rows=entity_rows
            ).to_dict()
            
            df_feat = pd.DataFrame(features)
            
            if self.feature_name in df_feat.columns:
                null_count = df_feat[self.feature_name].isna().sum()
                null_ratio = null_count / len(df_feat) if len(df_feat) > 0 else 0
                
                if null_ratio > self.threshold:
                    alert = {
                        "timestamp": datetime.now(),
                        "type": "high_null_ratio",
                        "message": f"{self.feature_name} null ratio: {null_ratio:.2%} (threshold: {self.threshold:.2%})",
                        "severity": "WARNING"
                    }
                    self.alerts.append(alert)
                    print(f"⚠️ ALERT: {alert['message']}")
                else:
                    print(f"✅ {self.feature_name} normal: {null_ratio:.2%} null values")
                    
        except Exception as e:
            alert = {
                "timestamp": datetime.now(),
                "type": "error",
                "message": f"Feature query error: {e}",
                "severity": "ERROR"
            }
            self.alerts.append(alert)
            print(f"❌ ERROR: {alert['message']}")
    
    def get_alerts(self):
        return pd.DataFrame(self.alerts)
    
    def clear_alerts(self):
        self.alerts = []
        print("✅ Alerts cleared")

real_customers = feast_df['customer_id'].dropna().unique()[:10].tolist()

print(f"Real customer ID samples: {real_customers}")

monitor = FeatureMonitor(
    store=store, 
    feature_view_name="transaction_features", 
    feature_name="amount", 
    threshold=0.3
)

test_entities = [{"customer_id": cust_id} for cust_id in real_customers]

print("\n=== Feature Monitoring Alert System ===")
monitor.check_feature_stats(test_entities)

print("\n=== Alert Log ===")
print(monitor.get_alerts())

print("\n✅ Step 14 completed - Monitoring system established")

Real customer ID samples: ['C1000000639', 'C1000001337', 'C1000001725', 'C1000002591', 'C1000003372', 'C1000003615', 'C1000004053', 'C1000004530', 'C1000005353', 'C1000005555']

=== Feature Monitoring Alert System ===
✅ amount normal: 0.00% null values

=== Alert Log ===
Empty DataFrame
Columns: []
Index: []

✅ Step 14 completed - Monitoring system established
